# NEU-CLS — reconstructed sensor-robustness evaluation

**Status: NOT EXECUTED.** This is a new reconstruction, **not** the original missing Kaggle notebook, and contains **no fabricated results**.

Requires the legitimate NEU-CLS class-directory dataset (official DOI: 10.6084/m9.figshare.28903550.v1), plus the exported trained ResNet-18 ONNX model from the project's own `cnn/train_cnn.py` / Hugging Face repository. Specify `NEU_CLS_ROOT` and `NEU_ONNX_PATH`.

Protocol: deterministic stratified seed-42 holdout (20% of images); original images vs Gaussian blur, JPEG degradation, and additive sensor noise. Class ordering and preprocessing match the project's declared ResNet-18 baseline. **This evaluates perturbed examples on the same fixed holdout; it is not an external production test.** Record model/data SHA-256 hashes before publishing.

In [ ]:
from pathlib import Path
from io import BytesIO
import hashlib, json, os
import numpy as np
import onnxruntime as ort
from PIL import Image, ImageFilter
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
DATA_ROOT = Path(os.environ.get("NEU_CLS_ROOT", "data/NEU-CLS"))
MODEL = Path(os.environ.get("NEU_ONNX_PATH", "models/neu_resnet18.onnx"))
if not DATA_ROOT.is_dir() or not MODEL.is_file():
    raise FileNotFoundError("Provide actual NEU-CLS class folders and the source-trained ONNX model; do not use generated data or fabricated results.")
paths = sorted(p for p in DATA_ROOT.rglob("*.jpg") if p.parent.name in CLASSES)
y = np.asarray([CLASSES.index(p.parent.name) for p in paths], dtype=np.int64)
if len(paths) != 1800 or set(y) != set(range(6)):
    raise ValueError(f"Expected the 1,800-image NEU-CLS benchmark with six classes, got {len(paths)} images; check root and class names.")
train_idx, test_idx = train_test_split(np.arange(len(paths)), test_size=.2, stratify=y, random_state=SEED)
assert len(test_idx) == 360
session = ort.InferenceSession(str(MODEL), providers=["CPUExecutionProvider"])
model_hash = hashlib.sha256(MODEL.read_bytes()).hexdigest()
print("Verified holdout:", len(test_idx), "model SHA-256:", model_hash)

In [ ]:
def image_to_tensor(img):
    img = img.convert("L").convert("RGB").resize((224,224), Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = (arr - np.array([.485,.456,.406], dtype=np.float32)) / np.array([.229,.224,.225], dtype=np.float32)
    return arr.transpose(2,0,1)[None].astype(np.float32)

def perturb(img, condition, seed):
    img = img.convert("L").convert("RGB")
    if condition == "original":
        return img
    if condition == "blur_radius_2":
        return img.filter(ImageFilter.GaussianBlur(radius=2))
    if condition == "jpeg_quality_35":
        buf = BytesIO()
        img.save(buf, "JPEG", quality=35)
        buf.seek(0)
        with Image.open(buf) as decoded:
            return decoded.convert("RGB").copy()
    if condition == "sensor_noise_sigma_28":
        rng = np.random.default_rng(seed)
        arr = np.asarray(img, dtype=np.float32)
        noise = rng.normal(0,28,size=(arr.shape[0],arr.shape[1],1))
        return Image.fromarray(np.clip(arr+noise,0,255).astype("uint8"), "RGB")
    raise ValueError(condition)

def infer(img):
    name = session.get_inputs()[0].name
    logits = session.run(None,{name:image_to_tensor(img)})[0]
    if logits.shape[-1] != len(CLASSES):
        raise ValueError(f"Expected six model logits, got {logits.shape}")
    return int(np.argmax(logits[0]))

In [ ]:
conditions = ["original","blur_radius_2","jpeg_quality_35","sensor_noise_sigma_28"]
measured = {}
for condition in conditions:
    truth,predicted = [],[]
    for idx in test_idx:
        path = paths[idx]
        stable_seed = int.from_bytes(hashlib.sha256(path.read_bytes()).digest()[:8],"little") + SEED
        with Image.open(path) as raw:
            prepared = perturb(raw, condition, stable_seed)
        truth.append(int(y[idx]))
        predicted.append(infer(prepared))
    measured[condition] = {
        "n":len(truth),
        "accuracy":float(accuracy_score(truth,predicted)),
        "macro_f1":float(f1_score(truth,predicted,average="macro")),
        "errors":int(np.sum(np.asarray(truth)!=np.asarray(predicted))),
    }
print(json.dumps(measured,indent=2))

In [ ]:
input_manifest = [
    {"relative_path":str(paths[i].relative_to(DATA_ROOT)).replace("\\","/"),
     "sha256":hashlib.sha256(paths[i].read_bytes()).hexdigest()}
    for i in sorted(test_idx)
]
report = {
  "status":"measured_from_provided_actual_data_and_model",
  "method":"from-scratch NEU-CLS fixed-holdout perturbation reconstruction, not the original Kaggle study",
  "original_source_doi":"10.6084/m9.figshare.28903550.v1",
  "model_sha256":model_hash,"seed":SEED,"classes":CLASSES,
  "test_images":input_manifest,"conditions":measured,
}
out=Path(os.environ.get("NEU_ROBUSTNESS_OUTPUT","research/outputs/neu_robustness_reconstruction.json"))
out.parent.mkdir(parents=True,exist_ok=True)
out.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Wrote measured output:",out,"(verify and cite before publication)")